<a href="https://colab.research.google.com/github/minseonju/Marine/blob/main/CAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [ ]:
df_normal = pd.read_csv('features_normal.csv')
df_attack = pd.read_csv('features_attack_merged.csv')

# 공격 유형별 분리 (평가 때 각각 탐지율 측정용)
df_rs     = df_attack[df_attack['label'] == 'reverse_shell'].drop(columns='label')
df_crypto = df_attack[df_attack['label'] == 'crypto_mining'].drop(columns='label')
df_attack = df_attack.drop(columns='label')

print(f'Normal  : {len(df_normal):,}행')
print(f'Attack  : {len(df_attack)}행')
print(f'  └ Reverse Shell : {len(df_rs)}행')
print(f'  └ Crypto Mining : {len(df_crypto)}행')

In [ ]:
common_cols = sorted(set(df_normal.columns) & set(df_attack.columns))

X_normal = df_normal[common_cols].values.astype(np.float32)
X_rs     = df_rs[common_cols].values.astype(np.float32)
X_crypto = df_crypto[common_cols].values.astype(np.float32)

# 분산 필터링 — Normal 기준 fit, 전체 동일 적용
selector = VarianceThreshold(threshold=1e-6)
selector.fit(X_normal)
mask     = selector.get_support()

X_normal = X_normal[:, mask]
X_rs     = X_rs[:, mask]
X_crypto = X_crypto[:, mask]
feat_cols = [common_cols[i] for i in range(len(common_cols)) if mask[i]]
INPUT_DIM = len(feat_cols)

print(f'피처 수 (필터 후) : {INPUT_DIM}개  ({len(common_cols) - INPUT_DIM}개 제거)')
print(f'Normal  샘플 : {len(X_normal):,}개')
print(f'RS      샘플 : {len(X_rs)}개')
print(f'Crypto  샘플 : {len(X_crypto)}개')

In [ ]:
np.random.seed(42)

# Normal 80/20 분리
idx     = np.random.permutation(len(X_normal))
n_train = int(len(X_normal) * 0.8)
X_train_normal = X_normal[idx[:n_train]]
X_test_normal  = X_normal[idx[n_train:]]

# 테스트셋: Normal 20% + 리버스셸 원본 전체 + 크립토마이닝 원본 전체
X_test = np.vstack([X_test_normal, X_rs, X_crypto])
y_true = np.array(
    [0] * len(X_test_normal) +
    [1] * len(X_rs) +
    [1] * len(X_crypto)
)

# 공격 유형 마스크
rs_mask     = np.array([False] * len(X_test_normal) +
                       [True]  * len(X_rs) +
                       [False] * len(X_crypto))
crypto_mask = np.array([False] * len(X_test_normal) +
                       [False] * len(X_rs) +
                       [True]  * len(X_crypto))

print(f'  Normal        : {len(X_test_normal):,}개')
print(f'  Reverse Shell : {len(X_rs)}개')
print(f'  Crypto Mining : {len(X_crypto)}개')
print(f'  합계          : {len(X_test):,}개')

In [ ]:
class CAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Flatten(),
            nn.Linear(16 * input_dim, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16 * input_dim),
            nn.Unflatten(1, (16, input_dim)),
            nn.ConvTranspose1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 1, kernel_size=3, padding=1)
        )
        self.latent_dim = latent_dim
        self.input_dim  = input_dim

    def forward(self, x):
        x   = x.unsqueeze(1)
        z   = self.encoder(x)
        out = self.decoder(z).squeeze(1)
        return out, z

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.MSELoss()
print(f'device: {device}')

In [ ]:
def evaluate(exp_name, y_true, y_pred, score_te, rs_mask, crypto_mask):
    nm_mask   = y_true == 0
    total_cr  = ((y_pred==1) & (y_true==1)).sum() / max((y_true==1).sum(), 1) * 100
    rs_cr     = ((y_pred==1) & rs_mask).sum()     / max(rs_mask.sum(), 1)     * 100
    crypto_cr = ((y_pred==1) & crypto_mask).sum() / max(crypto_mask.sum(), 1) * 100
    normal_nr = ((y_pred==0) & nm_mask).sum()     / max(nm_mask.sum(), 1)     * 100
    mf1       = f1_score(y_true, y_pred, average='macro')
    auc       = roc_auc_score(y_true, score_te)

    print(f'\n{"="*60}')
    print(f'[{exp_name}]')
    print(f'{"="*60}')
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Attack']))
    print(f'ROC-AUC            : {auc:.4f}')
    print(f'Macro F1           : {mf1:.4f}')
    print(f'─────────────────────────────')
    print(f'전체 Attack Recall : {total_cr:.1f}%')
    print(f'  Reverse Shell    : {rs_cr:.1f}%   ({rs_mask.sum()}개 중)')
    print(f'  Crypto Mining    : {crypto_cr:.1f}%   ({crypto_mask.sum()}개 중)')
    print(f'Normal Recall      : {normal_nr:.1f}%')

    return {
        'total_recall': total_cr, 'rs_recall': rs_cr,
        'crypto_recall': crypto_cr, 'normal_recall': normal_nr,
        'macro_f1': mf1, 'auc': auc
    }

print('평가 함수 정의 완료')

In [ ]:

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_normal).astype(np.float32)
X_te_s = scaler.transform(X_test).astype(np.float32)

train_t = torch.FloatTensor(X_tr_s).to(device)
loader  = DataLoader(TensorDataset(train_t), batch_size=64, shuffle=True)

cae     = CAE(INPUT_DIM, latent_dim=8).to(device)
opt     = optim.Adam(cae.parameters(), lr=1e-3, weight_decay=1e-5)
sch     = optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.5)
losses  = []

cae.train()
for epoch in range(50):
    loss_sum = 0
    for (batch,) in loader:
        opt.zero_grad()
        recon, _ = cae(batch)
        loss = criterion(recon, batch)
        loss.backward()
        opt.step()
        loss_sum += loss.item()
    avg = loss_sum / len(loader)
    losses.append(avg)
    sch.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch [{epoch+1:2d}/50]  Loss: {avg:.6f}')

In [ ]:
cae.eval()
with torch.no_grad():
    recon_tr, _ = cae(train_t)
    ae_tr = torch.mean((train_t - recon_tr) ** 2, dim=1).cpu().numpy()

    test_t = torch.FloatTensor(X_te_s).to(device)
    recon_te, _ = cae(test_t)
    ae_te = torch.mean((test_t - recon_te) ** 2, dim=1).cpu().numpy()

# 임계값 탐색
best_t, best_nr = None, -1
for pct in np.arange(80, 100, 0.5):
    t    = np.percentile(ae_tr, pct)
    pred = (ae_te >= t).astype(int)
    ar   = ((pred==1) & (y_true==1)).sum() / max((y_true==1).sum(), 1)
    nr   = ((pred==0) & (y_true==0)).sum() / max((y_true==0).sum(), 1)
    if ar >= 0.80 and nr > best_nr:
        best_nr, best_t = nr, t

if best_t is None:
    print('recall 목표 미달 → 95th percentile 기본값 사용')
    best_t = np.percentile(ae_tr, 95)

y_pred = (ae_te >= best_t).astype(int)
print(f'최적 임계값: {best_t:.4f}')

results = evaluate('CAE 단독 (비지도)', y_true, y_pred, ae_te, rs_mask, crypto_mask)

In [ ]:
nm_mask = (y_true == 0)

fig = plt.figure(figsize=(20, 5))
gs  = gridspec.GridSpec(1, 4, wspace=0.35)

# ── Training Loss ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(losses, color='steelblue', linewidth=2)
ax1.set_title('CAE Training Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.grid(True, alpha=0.3)

# ── Reconstruction Error 분포 ────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.hist(ae_te[nm_mask],     bins=60, alpha=0.5, density=True,
         label=f'Normal (n={nm_mask.sum()})', color='steelblue')
ax2.hist(ae_te[rs_mask],     bins=60, alpha=0.6, density=True,
         label=f'Reverse Shell (n={rs_mask.sum()})', color='tomato')
ax2.hist(ae_te[crypto_mask], bins=60, alpha=0.6, density=True,
         label=f'Crypto Mining (n={crypto_mask.sum()})', color='orange')
ax2.axvline(np.percentile(ae_tr, 95), color='gray',
            linestyle=':', linewidth=1.5, label='95th pct (train)')
ax2.set_title('Reconstruction Error Distribution', fontweight='bold')
ax2.set_xlabel('MSE Reconstruction Error')
ax2.set_ylabel('Density')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── ROC Curve ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
fpr, tpr, _ = roc_curve(y_true, ae_te)
ax3.plot(fpr, tpr, color='steelblue', linewidth=2,
         label=f'AUC = {results["auc"]:.4f}')
ax3.plot([0,1],[0,1], 'k--', linewidth=1, label='Random')
ax3.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax3.set_title('ROC Curve', fontweight='bold')
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# ── 임계값 탐색 (macro F1 기준) ──────────────────────────────
ax4 = fig.add_subplot(gs[3])
pcts = np.arange(80, 100, 0.5)
f1s  = []
for pct in pcts:
    t   = np.percentile(ae_tr, pct)
    pr  = (ae_te >= t).astype(int)
    f1s.append(f1_score(y_true, pr, average='macro'))
best_idx = int(np.argmax(f1s))
ax4.plot(pcts, f1s, color='steelblue', linewidth=2)
ax4.axvline(pcts[best_idx], color='tomato', linestyle='--', linewidth=2,
            label=f'Best={pcts[best_idx]:.1f}th\nF1={f1s[best_idx]:.4f}')
ax4.set_title('Threshold Search (macro F1)', fontweight='bold')
ax4.set_xlabel('Percentile')
ax4.set_ylabel('macro F1')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.suptitle('CAE 단독 (비지도) — Training Loss / Error Distribution / ROC / Threshold',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cae_analysis_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig2 = plt.figure(figsize=(18, 5))
gs2  = gridspec.GridSpec(1, 3, wspace=0.35)

# ── Anomaly Score 분포 ───────────────────────────────────────
ax5 = fig2.add_subplot(gs2[0])
ax5.hist(ae_te[nm_mask],     bins=40, alpha=0.5, density=True,
         label=f'Normal ({nm_mask.sum()})', color='steelblue')
ax5.hist(ae_te[rs_mask],     bins=40, alpha=0.6, density=True,
         label=f'Reverse Shell ({rs_mask.sum()})', color='tomato')
ax5.hist(ae_te[crypto_mask], bins=40, alpha=0.6, density=True,
         label=f'Crypto Mining ({crypto_mask.sum()})', color='orange')
ax5.axvline(best_t, color='black', linestyle='--', linewidth=2,
            label=f'Threshold={best_t:.3f}')
ax5.set_title('Anomaly Score Distribution', fontweight='bold')
ax5.set_xlabel('MSE Reconstruction Error (높을수록 이상)')
ax5.set_ylabel('Density')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

# ── 혼동 행렬 ────────────────────────────────────────────────
ax6 = fig2.add_subplot(gs2[1])
cm  = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax6,
            xticklabels=['Pred Normal', 'Pred Attack'],
            yticklabels=['Actual Normal', 'Actual Attack'],
            annot_kws={'size': 13})
ax6.set_title('Confusion Matrix', fontweight='bold')

# ── 공격 유형별 탐지율 ───────────────────────────────────────
ax7 = fig2.add_subplot(gs2[2])
categories = ['Reverse Shell', 'Crypto Mining', 'Total Attack']
recalls    = [results['rs_recall'], results['crypto_recall'], results['total_recall']]
colors     = ['tomato' if r < 80 else 'steelblue' for r in recalls]

bars = ax7.bar(categories, recalls, color=colors, edgecolor='white', width=0.5)
ax7.axhline(80, color='gray', linestyle='--', linewidth=1.5, label='80% 기준선')
for bar, recall in zip(bars, recalls):
    ax7.text(bar.get_x() + bar.get_width()/2, recall + 1.5,
             f'{recall:.1f}%', ha='center', va='bottom',
             fontsize=12, fontweight='bold')
ax7.set_ylim(0, 115)
ax7.set_title('Attack Detection Rate by Type', fontweight='bold')
ax7.set_ylabel('Detection Rate (%)')
ax7.legend(fontsize=9)
ax7.grid(True, alpha=0.3, axis='y')

plt.suptitle('CAE 단독 (비지도) — Score Distribution / Confusion Matrix / Detection Rate',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cae_analysis_2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 55)
print('최종 성능 수치 요약 (CAE 단독 — 완전 비지도)')
print('=' * 55)
print(classification_report(y_true, y_pred, target_names=['Normal', 'Attack']))
print(f'ROC-AUC            : {results["auc"]:.4f}')
print(f'최적 Threshold     : {best_t:.4f}')
print(f'Macro F1           : {results["macro_f1"]:.4f}')
print()
print('[공격 유형별 탐지율]')
for name, key in [('Reverse Shell', 'rs_recall'),
                  ('Crypto Mining', 'crypto_recall'),
                  ('전체 Attack  ', 'total_recall')]:
    r   = results[key]
    bar = '█' * int(r // 5)
    print(f'  {name} : {r:5.1f}%  {bar}')
print(f'  Normal 유지율  : {results["normal_recall"]:5.1f}%')